# Homework 05 - Data Storage

This notebook saves a DataFrame in CSV and Parquet formats, reloads both files, validates the results, and defines reusable read/write utility functions.

In [2]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv

In [3]:
HOMEWORK_ROOT = Path.cwd().parent
env_path = HOMEWORK_ROOT / ".env"

load_dotenv(env_path)

DATA_DIR_RAW = HOMEWORK_ROOT / os.getenv("DATA_DIR_RAW", "data/raw")
DATA_DIR_PROCESSED = HOMEWORK_ROOT / os.getenv("DATA_DIR_PROCESSED", "data/processed")

DATA_DIR_RAW.mkdir(parents=True, exist_ok=True)
DATA_DIR_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Homework root:", HOMEWORK_ROOT)
print("Raw data folder:", DATA_DIR_RAW)
print("Processed data folder:", DATA_DIR_PROCESSED)
print("Raw folder exists:", DATA_DIR_RAW.exists())
print("Processed folder exists:", DATA_DIR_PROCESSED.exists())

python-dotenv could not parse statement starting at line 2


Homework root: /Users/devampatel/Documents/NYU/homework/homework05
Raw data folder: /Users/devampatel/Documents/NYU/homework/homework05/data/raw
Processed data folder: /Users/devampatel/Documents/NYU/homework/homework05/data/processed
Raw folder exists: True
Processed folder exists: True


## 1. Create Sample Data

This section creates a small dataset to practice saving, loading, and validating storage formats.

In [4]:
loan_sample = pd.DataFrame({
    "applicant_id": [1001, 1002, 1003, 1004, 1005],
    "income": [55000, 72000, 41000, 88000, 62000],
    "loan_amount": [12000, 18000, 9000, 25000, 15000],
    "default_risk": ["low", "medium", "high", "low", "medium"]
})

loan_sample

,applicant_id,income,loan_amount,default_risk
0,1001,55000,12000,low
1,1002,72000,18000,medium
2,1003,41000,9000,high
3,1004,88000,25000,low
4,1005,62000,15000,medium


## 2. Save CSV and Parquet

This section saves the same DataFrame as a CSV file in `data/raw/` and as a Parquet file in `data/processed/`.

In [5]:
csv_path = DATA_DIR_RAW / "loan_sample.csv"
parquet_path = DATA_DIR_PROCESSED / "loan_sample.parquet"

loan_sample.to_csv(csv_path, index=False)
loan_sample.to_parquet(parquet_path, index=False)

print("Saved CSV to:", csv_path)
print("Saved Parquet to:", parquet_path)

Saved CSV to: /Users/devampatel/Documents/NYU/homework/homework05/data/raw/loan_sample.csv
Saved Parquet to: /Users/devampatel/Documents/NYU/homework/homework05/data/processed/loan_sample.parquet


## 3. Reload and Validate

This section reloads the CSV and Parquet files and checks that their shapes and key column data types match.

In [6]:
csv_loaded = pd.read_csv(csv_path)
parquet_loaded = pd.read_parquet(parquet_path)

print("Original shape:", loan_sample.shape)
print("CSV shape:", csv_loaded.shape)
print("Parquet shape:", parquet_loaded.shape)

print("CSV shape matches original:", csv_loaded.shape == loan_sample.shape)
print("Parquet shape matches original:", parquet_loaded.shape == loan_sample.shape)

Original shape: (5, 4)
CSV shape: (5, 4)
Parquet shape: (5, 4)
CSV shape matches original: True
Parquet shape matches original: True


In [7]:
def validate_storage(original_df, loaded_df, critical_columns):
    """Check shape and selected column dtypes after reloading data."""
    results = {
        "shape_matches": original_df.shape == loaded_df.shape,
        "dtypes_match": all(
            original_df[column].dtype == loaded_df[column].dtype
            for column in critical_columns
        )
    }
    return results

In [8]:
critical_columns = ["applicant_id", "income", "loan_amount", "default_risk"]

csv_validation = validate_storage(loan_sample, csv_loaded, critical_columns)
parquet_validation = validate_storage(loan_sample, parquet_loaded, critical_columns)

print("CSV validation:", csv_validation)
print("Parquet validation:", parquet_validation)

CSV validation: {'shape_matches': True, 'dtypes_match': True}
Parquet validation: {'shape_matches': True, 'dtypes_match': True}


## 4. Reusable Storage Utilities

This section defines `write_df` and `read_df` functions that choose the correct pandas method based on the file suffix.

In [9]:
def write_df(dataframe, path):
    """Write a DataFrame to CSV or Parquet based on the file suffix."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.suffix == ".csv":
        dataframe.to_csv(path, index=False)
    elif path.suffix == ".parquet":
        try:
            dataframe.to_parquet(path, index=False)
        except ImportError as error:
            raise ImportError(
                "Parquet support requires pyarrow or fastparquet. Install pyarrow and try again."
            ) from error
    else:
        raise ValueError("Unsupported file type. Use .csv or .parquet.")


def read_df(path):
    """Read a CSV or Parquet file based on the file suffix."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    if path.suffix == ".csv":
        return pd.read_csv(path)
    elif path.suffix == ".parquet":
        try:
            return pd.read_parquet(path)
        except ImportError as error:
            raise ImportError(
                "Parquet support requires pyarrow or fastparquet. Install pyarrow and try again."
            ) from error
    else:
        raise ValueError("Unsupported file type. Use .csv or .parquet.")

In [11]:
utility_csv_path = DATA_DIR_RAW / "loan_sample_utility.csv"
utility_parquet_path = DATA_DIR_PROCESSED / "loan_sample_utility.parquet"

write_df(loan_sample, utility_csv_path)
write_df(loan_sample, utility_parquet_path)

utility_csv_loaded = read_df(utility_csv_path)
utility_parquet_loaded = read_df(utility_parquet_path)

print("Utility CSV shape:", utility_csv_loaded.shape)
print("Utility Parquet shape:", utility_parquet_loaded.shape)

Utility CSV shape: (5, 4)
Utility Parquet shape: (5, 4)
